# 09 - TotalSegmentator Deep-Learning Skull and Brain Reconstruction

This notebook applies the pretrained **TotalSegmentator `total` CT task**, which
is based on nnU-Net, to this anonymized DICOM study. Inference is restricted to
the model's `brain` and `skull` output classes. The two neural-network masks are
then converted into separate and combined 3D triangle meshes.

Important distinctions:

- DICOM-to-NIfTI conversion is only an input-format conversion.
- Skull and brain segmentation comes exclusively from TotalSegmentator.
- No HU threshold or handcrafted fallback mask is used.
- Marching Cubes and Taubin smoothing create the display meshes *after* neural
  network segmentation; they do not decide which voxels are brain or skull.

TotalSegmentator is not a medical device, and these outputs are not intended
for diagnosis or clinical decision-making.

In [1]:
from pathlib import Path
import importlib.metadata as metadata
import json
import time

import dicom2nifti
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import plotly.graph_objects as go
import pydicom
import torch
import trimesh
from scipy import ndimage as ndi
from skimage import measure
from totalsegmentator.python_api import totalsegmentator

STARTED = time.time()
DATASET_ROOT = Path(r"F:\csstdy\Single_Patient_Study")
OUTPUT_DIR = Path(r"F:\csstdy\outputs\09_totalsegmentator_dl_skull_brain")
INPUT_DIR = OUTPUT_DIR / "input"
SEGMENTATION_DIR = OUTPUT_DIR / "segmentations"
MESH_DIR = OUTPUT_DIR / "meshes"
for directory in (INPUT_DIR, SEGMENTATION_DIR, MESH_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CT_NIFTI = INPUT_DIR / "CASE_001_brain_kernel_ct.nii.gz"
BRAIN_NIFTI = SEGMENTATION_DIR / "brain.nii.gz"
SKULL_NIFTI = SEGMENTATION_DIR / "skull.nii.gz"
REPORT_PATH = OUTPUT_DIR / "totalsegmentator_report.json"
SUMMARY_PATH = OUTPUT_DIR / "reconstruction_summary.json"
FORCE_RERUN = False
MESH_STEP_SIZE = 2

assert DATASET_ROOT.exists(), f"Dataset not found: {DATASET_ROOT}"
assert torch.cuda.is_available(), "CUDA GPU is required for this notebook execution."

print("TotalSegmentator:", metadata.version("TotalSegmentator"))
print("nnU-Net:", metadata.version("nnunetv2"))
print("PyTorch:", torch.__version__)
print("Inference device:", torch.cuda.get_device_name(0))
print("Output:", OUTPUT_DIR)

TotalSegmentator: 2.18.0
nnU-Net: 2.8.1
PyTorch: 2.14.0+cu130
Inference device: NVIDIA GeForce RTX 4060 Laptop GPU
Output: F:\csstdy\outputs\09_totalsegmentator_dl_skull_brain


## Select and convert the custom DICOM series

The soft-tissue reconstruction kernel is used as the single input CT for both
neural-network labels. The conversion preserves image geometry and HU values.

In [2]:
series_directories = sorted(
    path for path in DATASET_ROOT.rglob("BRAIN_HR36F") if path.is_dir()
)
if not series_directories:
    raise FileNotFoundError("No BRAIN_HR36F directory was found.")

DICOM_SERIES_DIR = series_directories[0]
dicom_paths = sorted(DICOM_SERIES_DIR.glob("*.dcm"))
if len(dicom_paths) < 3:
    raise RuntimeError(f"Too few DICOM files in {DICOM_SERIES_DIR}")

first = pydicom.dcmread(dicom_paths[0], stop_before_pixels=True, force=True)
print("Input series:", DICOM_SERIES_DIR.name)
print("DICOM slices:", len(dicom_paths))
print("Modality:", getattr(first, "Modality", ""))
print("Convolution kernel:", getattr(first, "ConvolutionKernel", ""))

if FORCE_RERUN or not CT_NIFTI.exists():
    conversion = dicom2nifti.dicom_series_to_nifti(
        str(DICOM_SERIES_DIR), str(CT_NIFTI), reorient_nifti=True
    )
    print("Created:", CT_NIFTI)
else:
    print("Using cached NIfTI input:", CT_NIFTI)

ct_image = nib.load(CT_NIFTI)
print("NIfTI shape:", ct_image.shape)
print("NIfTI voxel sizes (mm):", tuple(round(v, 4) for v in ct_image.header.get_zooms()[:3]))

Input series: BRAIN_HR36F
DICOM slices: 227
Modality: CT
Convolution kernel: ['Hr36f', '4']


Created: F:\csstdy\outputs\09_totalsegmentator_dl_skull_brain\input\CASE_001_brain_kernel_ct.nii.gz
NIfTI shape: (512, 512, 227)
NIfTI voxel sizes (mm): (np.float32(0.5557), np.float32(0.5557), np.float32(0.75))


## Run pretrained TotalSegmentator nnU-Net inference

The open `total` CT task contains the `brain` and `skull` classes. `roi_subset`
restricts inference and output to those two network labels. Existing valid mask
files are reused on later notebook runs; set `FORCE_RERUN=True` to infer again.

In [3]:
def valid_mask_file(path):
    if not path.exists():
        return False
    image = nib.load(path)
    return image.shape == ct_image.shape and np.asanyarray(image.dataobj).max() > 0


model_ran_now = FORCE_RERUN or not (
    valid_mask_file(BRAIN_NIFTI) and valid_mask_file(SKULL_NIFTI)
)
inference_started = time.time()

if model_ran_now:
    print("Starting full-resolution TotalSegmentator inference...")
    totalsegmentator(
        input=CT_NIFTI,
        output=SEGMENTATION_DIR,
        task="total",
        roi_subset=["brain", "skull"],
        device="gpu",
        ml=False,
        quiet=False,
        verbose=False,
        report=REPORT_PATH,
    )
else:
    print("Using cached TotalSegmentator brain and skull predictions.")

inference_seconds = time.time() - inference_started
if not valid_mask_file(BRAIN_NIFTI):
    raise RuntimeError(f"Missing or invalid TotalSegmentator brain mask: {BRAIN_NIFTI}")
if not valid_mask_file(SKULL_NIFTI):
    raise RuntimeError(f"Missing or invalid TotalSegmentator skull mask: {SKULL_NIFTI}")

print("Brain prediction:", BRAIN_NIFTI)
print("Skull prediction:", SKULL_NIFTI)
print("Inference executed now:", model_ran_now)
print("This-cell inference/cache time:", f"{inference_seconds:.1f} seconds")

Starting full-resolution TotalSegmentator inference...

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

TotalSegmentator sends anonymous usage statistics. If you want to disable it check the documentation.


Downloading:   0%|          | 0.00/234M [00:00<?, ?B/s]

Downloading:   0%|          | 655k/234M [00:00<00:37, 6.29MB/s]

Downloading:   1%|          | 2.88M/234M [00:00<00:14, 15.5MB/s]

Downloading:   2%|▏         | 5.24M/234M [00:00<00:12, 19.0MB/s]

Downloading:   3%|▎         | 7.21M/234M [00:00<00:12, 18.5MB/s]

... (1,740 outputs left out)

Saving segmentations...


  Saved in 8.46s


Run report written to F:\csstdy\outputs\09_totalsegmentator_dl_skull_brain\totalsegmentator_report.json


Brain prediction: F:\csstdy\outputs\09_totalsegmentator_dl_skull_brain\segmentations\brain.nii.gz
Skull prediction: F:\csstdy\outputs\09_totalsegmentator_dl_skull_brain\segmentations\skull.nii.gz
Inference executed now: True
This-cell inference/cache time: 297.0 seconds


## Validate the two neural-network masks

In [4]:
brain_image = nib.load(BRAIN_NIFTI)
skull_image = nib.load(SKULL_NIFTI)

if brain_image.shape != ct_image.shape or skull_image.shape != ct_image.shape:
    raise RuntimeError("TotalSegmentator masks do not match the input CT shape.")
if not np.allclose(brain_image.affine, ct_image.affine, atol=1e-4):
    raise RuntimeError("Brain-mask affine does not match the input CT.")
if not np.allclose(skull_image.affine, ct_image.affine, atol=1e-4):
    raise RuntimeError("Skull-mask affine does not match the input CT.")

ct_volume = np.asanyarray(ct_image.dataobj).astype(np.float32)
brain_mask = np.asanyarray(brain_image.dataobj) > 0
skull_mask = np.asanyarray(skull_image.dataobj) > 0
voxel_volume_mm3 = float(abs(np.linalg.det(ct_image.affine[:3, :3])))

brain_components = int(ndi.label(brain_mask)[1])
skull_components = int(ndi.label(skull_mask)[1])
brain_volume_ml = float(brain_mask.sum() * voxel_volume_mm3 / 1000.0)
skull_volume_ml = float(skull_mask.sum() * voxel_volume_mm3 / 1000.0)

print("Brain voxels:", f"{int(brain_mask.sum()):,}")
print("Skull voxels:", f"{int(skull_mask.sum()):,}")
print("Brain connected components (raw prediction):", brain_components)
print("Skull connected components (raw prediction):", skull_components)
print("Brain volume:", f"{brain_volume_ml:.1f} mL")
print("Skull-label volume:", f"{skull_volume_ml:.1f} mL")

Brain voxels: 4,975,324
Skull voxels: 2,496,034
Brain connected components (raw prediction): 1
Skull connected components (raw prediction): 9
Brain volume: 1152.5 mL
Skull-label volume: 578.2 mL


## Check predicted masks against the CT

In [5]:
occupied_slices = np.flatnonzero(
    np.any(brain_mask | skull_mask, axis=(0, 1))
)
preview_slices = np.linspace(
    occupied_slices[0], occupied_slices[-1], 9, dtype=int
)

overlay_figure, axes = plt.subplots(3, 3, figsize=(11, 11), dpi=130)
for axis, z in zip(axes.ravel(), preview_slices):
    ct_slice = np.rot90(ct_volume[:, :, z])
    brain_slice = np.rot90(brain_mask[:, :, z])
    skull_slice = np.rot90(skull_mask[:, :, z])
    axis.imshow(ct_slice, cmap="gray", vmin=-20, vmax=100)
    if skull_slice.any():
        axis.contour(skull_slice, levels=[0.5], colors=["#ffd166"], linewidths=0.8)
    if brain_slice.any():
        axis.contour(brain_slice, levels=[0.5], colors=["#ff4e50"], linewidths=1.0)
    axis.set_title(f"NIfTI axial slice {z}")
    axis.axis("off")

overlay_figure.suptitle("TotalSegmentator predictions: skull (yellow), brain (red)")
overlay_figure.tight_layout()
OVERLAY_PATH = OUTPUT_DIR / "CASE_001_totalsegmentator_overlay.png"
overlay_figure.savefig(OVERLAY_PATH, bbox_inches="tight")
plt.show()

C:\Users\Pranjal\AppData\Local\Temp\ipykernel_34660\2412871434.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Convert predicted masks to 3D meshes

Marching Cubes operates only on TotalSegmentator's predicted masks. Mesh
smoothing is a visualization step and does not change the saved NIfTI labels.

In [6]:
def prediction_to_mesh(mask, affine, smoothing_iterations):
    padded = np.pad(mask.astype(np.uint8), 1, mode="constant")
    vertices_voxel, faces, _normals, _values = measure.marching_cubes(
        padded,
        level=0.5,
        step_size=MESH_STEP_SIZE,
        allow_degenerate=False,
    )
    vertices_voxel -= 1.0
    vertices_world = nib.affines.apply_affine(affine, vertices_voxel).astype(np.float32)
    mesh = trimesh.Trimesh(
        vertices=vertices_world, faces=faces, process=True, validate=True
    )
    mesh.remove_unreferenced_vertices()
    trimesh.smoothing.filter_taubin(
        mesh, lamb=0.5, nu=0.525, iterations=smoothing_iterations
    )
    mesh.fix_normals(multibody=True)
    return mesh


skull_mesh = prediction_to_mesh(skull_mask, skull_image.affine, 5)
brain_mesh = prediction_to_mesh(brain_mask, brain_image.affine, 10)
skull_mesh.visual.face_colors = [232, 214, 179, 205]
brain_mesh.visual.face_colors = [217, 155, 145, 255]

print("Skull mesh:", f"{len(skull_mesh.vertices):,} vertices,", f"{len(skull_mesh.faces):,} triangles")
print("Brain mesh:", f"{len(brain_mesh.vertices):,} vertices,", f"{len(brain_mesh.faces):,} triangles")

Skull mesh: 129,539 vertices, 259,190 triangles
Brain mesh: 59,054 vertices, 118,104 triangles


## Export and show the combined TotalSegmentator reconstruction

In [7]:
paths = {
    "brain_ply": MESH_DIR / "CASE_001_totalsegmentator_brain.ply",
    "brain_stl": MESH_DIR / "CASE_001_totalsegmentator_brain.stl",
    "brain_glb": MESH_DIR / "CASE_001_totalsegmentator_brain.glb",
    "skull_ply": MESH_DIR / "CASE_001_totalsegmentator_skull.ply",
    "skull_stl": MESH_DIR / "CASE_001_totalsegmentator_skull.stl",
    "skull_glb": MESH_DIR / "CASE_001_totalsegmentator_skull.glb",
    "combined_glb": MESH_DIR / "CASE_001_totalsegmentator_skull_and_brain.glb",
    "combined_html": OUTPUT_DIR / "CASE_001_totalsegmentator_skull_and_brain.html",
    "skull_html": OUTPUT_DIR / "CASE_001_totalsegmentator_skull_only.html",
    "brain_html": OUTPUT_DIR / "CASE_001_totalsegmentator_brain_only.html",
}

brain_mesh.export(paths["brain_ply"])
brain_mesh.export(paths["brain_stl"])
skull_mesh.export(paths["skull_ply"])
skull_mesh.export(paths["skull_stl"])
paths["brain_glb"].write_bytes(trimesh.Scene(brain_mesh).export(file_type="glb"))
paths["skull_glb"].write_bytes(trimesh.Scene(skull_mesh).export(file_type="glb"))

combined_scene = trimesh.Scene()
combined_scene.add_geometry(skull_mesh.copy(), node_name="skull", geom_name="skull")
combined_scene.add_geometry(brain_mesh.copy(), node_name="brain", geom_name="brain")
paths["combined_glb"].write_bytes(combined_scene.export(file_type="glb"))

skull_vertices = np.asarray(skull_mesh.vertices)
skull_faces = np.asarray(skull_mesh.faces)
brain_vertices = np.asarray(brain_mesh.vertices)
brain_faces = np.asarray(brain_mesh.faces)

combined_figure = go.Figure([
    go.Mesh3d(
        x=skull_vertices[:, 0], y=skull_vertices[:, 1], z=skull_vertices[:, 2],
        i=skull_faces[:, 0], j=skull_faces[:, 1], k=skull_faces[:, 2],
        name="TotalSegmentator skull", color="#e8d6b3", opacity=0.24,
        flatshading=False, hoverinfo="skip",
        lighting=dict(ambient=0.48, diffuse=0.65, specular=0.16, roughness=0.78),
    ),
    go.Mesh3d(
        x=brain_vertices[:, 0], y=brain_vertices[:, 1], z=brain_vertices[:, 2],
        i=brain_faces[:, 0], j=brain_faces[:, 1], k=brain_faces[:, 2],
        name="TotalSegmentator brain", color="#d99b91", opacity=1.0,
        flatshading=False, hoverinfo="skip",
        lighting=dict(ambient=0.44, diffuse=0.72, specular=0.18, roughness=0.78),
    ),
])

def style_figure(figure, title):
    figure.update_layout(
        title=title,
        template="plotly_dark",
        width=950,
        height=780,
        margin=dict(l=0, r=0, t=60, b=0),
        legend=dict(x=0.02, y=0.98),
        scene=dict(
            aspectmode="data",
            xaxis_title="Patient R-L (mm)",
            yaxis_title="Patient A-P (mm)",
            zaxis_title="Patient I-S (mm)",
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False),
            camera=dict(eye=dict(x=1.55, y=-1.55, z=1.15)),
        ),
    )
    return figure


style_figure(combined_figure, "CASE_001 - TotalSegmentator Skull and Brain")
combined_figure.write_html(paths["combined_html"], include_plotlyjs=True, full_html=True)
combined_figure.show()

## Skull-only neural-network reconstruction

In [8]:
skull_figure = go.Figure([go.Mesh3d(
    x=skull_vertices[:, 0], y=skull_vertices[:, 1], z=skull_vertices[:, 2],
    i=skull_faces[:, 0], j=skull_faces[:, 1], k=skull_faces[:, 2],
    name="TotalSegmentator skull", color="#e8d6b3", opacity=1.0,
    flatshading=False, hoverinfo="skip",
    lighting=dict(ambient=0.42, diffuse=0.72, specular=0.20, roughness=0.72),
)])
style_figure(skull_figure, "CASE_001 - TotalSegmentator Skull")
skull_figure.write_html(paths["skull_html"], include_plotlyjs=True, full_html=True)
skull_figure.show()

## Brain-only neural-network reconstruction

In [9]:
brain_figure = go.Figure([go.Mesh3d(
    x=brain_vertices[:, 0], y=brain_vertices[:, 1], z=brain_vertices[:, 2],
    i=brain_faces[:, 0], j=brain_faces[:, 1], k=brain_faces[:, 2],
    name="TotalSegmentator brain", color="#d99b91", opacity=1.0,
    flatshading=False, hoverinfo="skip",
    lighting=dict(ambient=0.44, diffuse=0.72, specular=0.18, roughness=0.78),
)])
style_figure(brain_figure, "CASE_001 - TotalSegmentator Brain")
brain_figure.write_html(paths["brain_html"], include_plotlyjs=True, full_html=True)
brain_figure.show()

## Reproducibility summary

In [10]:
summary = {
    "segmentation_method": "TotalSegmentator pretrained nnU-Net inference only",
    "totalsegmentator_version": metadata.version("TotalSegmentator"),
    "nnunet_version": metadata.version("nnunetv2"),
    "pytorch_version": torch.__version__,
    "device": torch.cuda.get_device_name(0),
    "task": "total",
    "roi_subset": ["brain", "skull"],
    "input_series": DICOM_SERIES_DIR.name,
    "input_dicom_slices": len(dicom_paths),
    "input_nifti": str(CT_NIFTI),
    "model_ran_during_this_execution": bool(model_ran_now),
    "inference_or_cache_seconds": inference_seconds,
    "brain_mask": {
        "path": str(BRAIN_NIFTI),
        "voxels": int(brain_mask.sum()),
        "volume_ml": brain_volume_ml,
        "raw_connected_components": brain_components,
    },
    "skull_mask": {
        "path": str(SKULL_NIFTI),
        "voxels": int(skull_mask.sum()),
        "volume_ml": skull_volume_ml,
        "raw_connected_components": skull_components,
    },
    "brain_mesh": {
        "vertices": int(len(brain_mesh.vertices)),
        "triangles": int(len(brain_mesh.faces)),
        "watertight": bool(brain_mesh.is_watertight),
    },
    "skull_mesh": {
        "vertices": int(len(skull_mesh.vertices)),
        "triangles": int(len(skull_mesh.faces)),
        "watertight": bool(skull_mesh.is_watertight),
    },
    "outputs": {name: str(path) for name, path in paths.items()},
    "overlay_path": str(OVERLAY_PATH),
    "totalsegmentator_report": str(REPORT_PATH),
    "total_notebook_seconds": time.time() - STARTED,
    "clinical_use": False,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

{
  "segmentation_method": "TotalSegmentator pretrained nnU-Net inference only",
  "totalsegmentator_version": "2.18.0",
  "nnunet_version": "2.8.1",
  "pytorch_version": "2.14.0+cu130",
  "device": "NVIDIA GeForce RTX 4060 Laptop GPU",
  "task": "total",
  "roi_subset": [
    "brain",
    "skull"
  ],
  "input_series": "BRAIN_HR36F",
  "input_dicom_slices": 227,
  "input_nifti": "F:\\csstdy\\outputs\\09_totalsegmentator_dl_skull_brain\\input\\CASE_001_brain_kernel_ct.nii.gz",
  "model_ran_during_this_execution": true,
  "inference_or_cache_seconds": 296.9823977947235,
  "brain_mask": {
    "path": "F:\\csstdy\\outputs\\09_totalsegmentator_dl_skull_brain\\segmentations\\brain.nii.gz",
    "voxels": 4975324,
    "volume_ml": 1152.485433598391,
    "raw_connected_components": 1
  },
  "skull_mask": {
    "path": "F:\\csstdy\\outputs\\09_totalsegmentator_dl_skull_brain\\segmentations\\skull.nii.gz",
    "voxels": 2496034,
    "volume_ml": 578.1820092050942,
    "raw_connected_components":